# Gabarito — Módulo 1: Word Embeddings

In [1]:
import re
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

sms_data = [
    ("ham", "Hey, are we still on for lunch tomorrow?"),
    ("ham", "I'll call you when I get home from work."),
    ("ham", "Can you send me the notes from today's class?"),
    ("ham", "Happy birthday! Hope you have an amazing day."),
    ("ham", "Running a bit late, be there in 10 minutes."),
    ("ham", "Thanks for the ride yesterday, really appreciated."),
    ("ham", "Don't forget the meeting tomorrow morning."),
    ("ham", "Can we reschedule our meeting to tomorrow?"),
    ("ham", "I'll be at the meeting, see you tomorrow."),
    ("ham", "Thanks so much, talk to you tomorrow."),
    ("ham", "Let's grab lunch after the meeting tomorrow."),
    ("ham", "Sorry I missed your call earlier, call me back."),
    ("ham", "Can you call me when you get a chance?"),
    ("ham", "I'll call the doctor to book an appointment."),
    ("ham", "Mom said dinner is ready, come home now."),
    ("ham", "See you at the gym later tonight."),
    ("ham", "Traffic is bad, I might be late for work."),
    ("ham", "Can you pick up the kids from school today?"),
    ("ham", "I forgot my notes at home, can you scan them?"),
    ("ham", "Movie night this weekend? Let me know."),
    ("ham", "Coffee tomorrow morning before work?"),
    ("ham", "Thanks for helping me with the project today."),
    ("ham", "The project meeting got moved to tomorrow."),
    ("ham", "Good night, talk to you tomorrow."),
    ("ham", "Congrats on the new job, so happy for you."),
    ("ham", "Can we do groceries together this weekend?"),
    ("ham", "I'll be home late tonight, don't wait for dinner."),
    ("ham", "Thanks again for everything, means a lot."),
    ("ham", "Let's plan the weekend trip, call me tonight."),
    ("ham", "Dad wants to know if you're coming home tomorrow."),
    ("ham", "Class got cancelled, no notes needed today."),
    ("ham", "I'll bring the notes to the meeting tomorrow."),
    ("ham", "Happy to help, just call me anytime."),
    ("ham", "See you tomorrow at the usual coffee place."),
    ("ham", "The doctor's appointment is confirmed for tomorrow."),
    ("ham", "Sorry for the late reply, was in a meeting."),
    ("ham", "Can you send the project file before tomorrow?"),
    ("ham", "Thanks for lunch, let's do it again soon."),
    ("ham", "I'll pick you up for the gym tomorrow morning."),
    ("ham", "Meeting notes are attached, check before tomorrow."),
    ("ham", "Happy weekend! See you at the gym."),
    ("ham", "Call me back when you're free, nothing urgent."),
    ("ham", "Thanks for the birthday wishes, means a lot."),
    ("ham", "Let's catch up over coffee this weekend."),
    ("ham", "I'm at work, will call you after the meeting."),
    ("ham", "Can you check the notes and call me tonight?"),
    ("ham", "Dinner at mom's tomorrow, don't be late."),
    ("ham", "Thanks for covering my shift today."),
    ("ham", "See you tomorrow, drive safe."),
    ("ham", "The meeting tomorrow is confirmed for 10am."),
    ("ham", "Congrats again, the whole team is proud."),
    ("ham", "Can we push the call to tomorrow afternoon?"),
    ("ham", "I'll send the notes right after the meeting."),
    ("ham", "Thanks for the coffee this morning."),
    ("ham", "Let's meet tomorrow to finish the project."),
    ("ham", "Good luck with the appointment tomorrow."),
    ("ham", "Call me tomorrow, I have some news to share."),
    ("ham", "Thanks for picking up the kids today."),
    ("ham", "See you at the meeting, bring your notes."),
    ("ham", "Home now, dinner will be ready soon."),
    ("ham", "Can't wait for the weekend trip, thanks for planning."),
    ("ham", "Sorry, stuck in traffic, call you when I'm home."),
    ("spam", "URGENT! You have won a free prize, claim now!"),
    ("spam", "Congratulations! You've been selected for a free cash award."),
    ("spam", "WIN a guaranteed cash prize, text WIN to claim now."),
    ("spam", "URGENT! Your mobile number has won a free prize, call now."),
    ("spam", "Free entry to win a cash prize, click the link now."),
    ("spam", "Claim your free cash prize now, urgent reply needed."),
    ("spam", "You have been selected to win a free voucher, claim now."),
    ("spam", "URGENT! Reply now to claim your free cash prize."),
    ("spam", "Winner! You've won a free cash award, text CLAIM now."),
    ("spam", "Free cash prize waiting, call now to claim urgent offer."),
    ("spam", "Exclusive offer: claim your free prize now, urgent!"),
    ("spam", "URGENT! Limited time, claim your free cash now."),
    ("spam", "Congratulations winner! Free cash prize, reply now to claim."),
    ("spam", "Your account has won a free prize, click now to claim."),
    ("spam", "Text WIN now for a chance to claim free cash prize."),
    ("spam", "URGENT offer! Free prize guaranteed, call now to claim."),
    ("spam", "You are a winner! Claim your free cash prize urgent."),
    ("spam", "Free voucher waiting, urgent reply to claim cash prize."),
    ("spam", "Congratulations! Urgent, claim your guaranteed prize now."),
    ("spam", "WIN free cash now, click link, urgent claim required."),
    ("spam", "URGENT! You have a free prize, text CLAIM to collect now."),
    ("spam", "Selected winner, free cash prize, call urgent now."),
    ("spam", "Claim now! Free prize and cash bonus, urgent offer."),
    ("spam", "URGENT! Free cash award waiting, reply CLAIM now."),
    ("spam", "Congratulations, you win! Claim your free prize urgent now."),
    ("ham", "Are you free tonight for dinner?"),
    ("ham", "Congrats on the win, let's celebrate this weekend!"),
    ("ham", "I'll text you the address now."),
    ("ham", "Call me back urgent, mom needs you home."),
    ("ham", "Can I get your number to text you later?"),
    ("ham", "Free tomorrow afternoon? Let's catch up."),
    ("ham", "Great news, call me now, I'm so excited!"),
    ("spam", "Reply now to secure your exclusive reward before it expires."),
    ("spam", "Your account is due a bonus, confirm today to receive it."),
    ("spam", "Act fast, this offer expires today, don't miss out."),
    ("spam", "You are eligible for a special gift, respond immediately."),
    ("spam", "Final notice: your reward is ready, confirm to collect."),
    ("spam", "Selected for an exclusive deal, confirm today to receive."),
]

sms = pd.DataFrame(sms_data, columns=["label", "text"])
sms.head()

,label,text
0,ham,"Hey, are we still on for lunch tomorrow?"
1,ham,I'll call you when I get home from work.
2,ham,Can you send me the notes from today's class?
3,ham,Happy birthday! Hope you have an amazing day.
4,ham,"Running a bit late, be there in 10 minutes."


In [2]:
# 1.1
print(sms.shape)
sms["label"].value_counts()

(100, 2)


label
ham     69
spam    31
Name: count, dtype: int64

In [3]:
# 1.2
sms["clean_text"] = sms["text"].str.lower().str.replace(r"[^a-z\s]", "", regex=True)
sms[["text", "clean_text"]].head()

,text,clean_text
0,"Hey, are we still on for lunch tomorrow?",hey are we still on for lunch tomorrow
1,I'll call you when I get home from work.,ill call you when i get home from work
2,Can you send me the notes from today's class?,can you send me the notes from todays class
3,Happy birthday! Hope you have an amazing day.,happy birthday hope you have an amazing day
4,"Running a bit late, be there in 10 minutes.",running a bit late be there in minutes


In [4]:
# 1.3
tokens = sms["clean_text"].str.split()

all_words = sorted(set(w for msg in tokens for w in msg))
word2idx = {"<PAD>": 0, "<UNK>": 1}
for w in all_words:
    word2idx[w] = len(word2idx)

vocab_size = len(word2idx)
print("Vocabulário:", vocab_size, "palavras")

Vocabulário: 225 palavras


In [5]:
# 1.4
window = 2
pairs = []
for msg in tokens:
    idxs = [word2idx[w] for w in msg]
    for i, target in enumerate(idxs):
        start = max(0, i - window)
        end = min(len(idxs), i + window + 1)
        for j in range(start, end):
            if j != i:
                pairs.append((target, idxs[j]))

print("Total de pares skip-gram:", len(pairs))

Total de pares skip-gram: 2680


In [6]:
# 1.5
targets = torch.tensor([p[0] for p in pairs], dtype=torch.long)
contexts = torch.tensor([p[1] for p in pairs], dtype=torch.long)

embed_dim = 32

class SkipGram(nn.Module):
    def __init__(self, vocab_size, embed_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.linear = nn.Linear(embed_dim, vocab_size)

    def forward(self, x):
        return self.linear(self.embedding(x))

model = SkipGram(vocab_size, embed_dim)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

for epoch in range(200):
    optimizer.zero_grad()
    logits = model(targets)
    loss = loss_fn(logits, contexts)
    loss.backward()
    optimizer.step()
    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch + 1}, loss={loss.item():.4f}")

Epoch 50, loss=3.1452
Epoch 100, loss=2.6486
Epoch 150, loss=2.5733
Epoch 200, loss=2.5585


In [7]:
# 1.6
embedding_matrix = model.embedding.weight.detach().numpy()
idx2word = {i: w for w, i in word2idx.items()}

def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9)

def most_similar(word, topn=5):
    target_vec = embedding_matrix[word2idx[word]]
    sims = []
    for w, idx in word2idx.items():
        if w in ("<PAD>", "<UNK>", word):
            continue
        sims.append((w, cosine_sim(target_vec, embedding_matrix[idx])))
    sims.sort(key=lambda x: x[1], reverse=True)
    return sims[:topn]

print("most similar to 'free':", most_similar("free"))
print("most similar to 'meeting':", most_similar("meeting"))

most similar to 'free': [('cash', np.float32(0.4943219)), ('chance', np.float32(0.43028963)), ('still', np.float32(0.4298521)), ('call', np.float32(0.42785174)), ('link', np.float32(0.39305845))]
most similar to 'meeting': [('winner', np.float32(0.43263066)), ('our', np.float32(0.3931851)), ('share', np.float32(0.36165646)), ('happy', np.float32(0.3453101)), ('right', np.float32(0.3438235))]


In [8]:
# 1.7
sms[["label", "text", "clean_text"]].to_csv("sms_clean.csv", index=False)

with open("vocab.json", "w") as f:
    json.dump(word2idx, f)

np.save("embedding_matrix.npy", embedding_matrix)
print("Salvo: sms_clean.csv, vocab.json, embedding_matrix.npy")

Salvo: sms_clean.csv, vocab.json, embedding_matrix.npy
